# Incompressible LES: NACA4412 with kEqnFjord

## Preamble

In [1]:
# Standard Library
from pathlib import Path
import foamnordic as fno
import onsaemiro as osm

### Directories

In [2]:
# Case Type and Name
CASE_TYPE = "les"
CASE_NAME = "NACA4412"
CATEGORY = "incompressible"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
OUTPUT_DIR = MAIN_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# DataGraph Configuration
FIG_X, FIG_Y = osm.figsize()
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette()
osm.set_style()

In [4]:
# HPC configuration
ACCOUNT = "<allocation-account>" # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 16
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

## Example - Closure Modelling

### k-equation closure

In [5]:
# k-equation function for subgrid-scale turbulence modelling
def keqn_function(k, velocity_grad, nu, delta, C_k=0.094, C_e=1.048):
    k_positive = fno.Math.maximum(left=k, right=1.0e-10)
    sqrt_k = fno.Math.sqrt(value=k_positive)
    strain = fno.Math.dev(value=2.0 * fno.Math.symm(value=velocity_grad))

    nut = C_k * delta * sqrt_k
    production = nut * fno.Math.ddot(left=velocity_grad, right=strain)
    dissipation = C_e * sqrt_k / delta

    k_new = fno.Field.transport(
        value=k,
        diffusivity=nu + nut,
        source=production,
        implicit_sink=dissipation,
        lower_bound=1.0e-10,
    )

    return {
        "k": k_new,
        "nut": C_k * delta * fno.Math.sqrt(value=k_new),
    }

In [6]:
# Define the k-equation closure
keqn_closure = fno.Closure(
    name="kEqnFjord",
    operator=fno.Operator.function(function=keqn_function),
    inputs={
        "k": fno.Field(name="k"),
        "velocity_grad": fno.Field.grad(value="U"),
        "nu": fno.Field(name="nu"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "k": fno.Field(name="k"),
        "nut": fno.Field(name="nut"),
    },
    key=None,
)

### Case Definition

In [7]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
    ranks=N_TASKS,
    integration=None,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [8]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    closures=(keqn_closure,),
    transforms=(),
    observations=(),
    placement=fno.Attached(model_cpus_per_node=None, data_path="auto"),
    scheduler=scheduler,
    name=None,
    verbose=True,
    numerics="convenient",
)

Resource,OpenFOAM,Model,Allocation
Processes,16 tasks,1 host(s),1 node(s)
CPU,16 cores,1 cores,17 cores
Memory,32 GiB,2 GiB,34.5 GiB
Field programs,-,1,1
SHM,-,~512 MiB,~512 MiB


In [9]:
# Launch the workload
run = longship.launch(
    readiness_timeout=120.0,
    termination_grace=30.0,
    orphan_timeout=30.0,
    start_timeout=900.0,
    verbose=None,
    profile_every=0,
    receive_memory_bytes=None,
)

[FoamNordic] kEqnFjord execution lane: native OpenFOAM.
[FoamNordic] Preparing mesh with blockMesh: NACA4412
[FoamNordic] Mesh is ready: NACA4412
[FoamNordic] Sailing submitted with Job ID: 1290373
[FoamNordic] Sailing has launched with Job ID: 1290373
[FoamNordic] Sailing started at: 2026-09-13T15:44:06
[FoamNordic] Sailing in background: NACA4412


In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact", display=True);

Job ID,Name,Status,Partition,Node,Elapsed
1290373,NACA4412,succeeded,small,rc5136,00:01:00


### Post-processing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p", "nut"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)

U shape: (89728, 3)
p shape: (89728,)


Field,Min,Max,Mean,Std,RMS
U,5.099908e-06,1.820225e+00,8.266874e-01,4.611462e-01,9.466087e-01
p,-1.259706e+00,5.414191e-01,-2.104359e-01,3.377630e-01,3.979537e-01
nut,2.200713e-07,5.923256e-04,7.610217e-06,3.587218e-05,3.667054e-05
